# Monster · Colab GPU Worker

Monster 源码、客户端、协议与部署入口统一维护在仓库的 `packages/monster`。
本 Notebook 只负责配置、安装指定部署包及启动，不包含 Base64 算法代码。

**方式一（最简，推荐）：** 直接在 GitHub 点击 **Open in Colab** 打开本 Notebook，选择 G4 GPU 后依次执行单元格 1 → 2 → 3 → 4。单元格 1 默认直接从 GitHub 拉取最新代码，无需手动打包或上传 `.tar.gz`。启动后可执行单元格 5 实时查看训练与任务进度（当前阶段、轮次、批次、loss、GPU 状态与预计剩余时间）。

**方式二（本地调试归档）：** 在本机运行 `npm run monster:package`，把生成的 `.tar.gz` 上传至 Drive 的 `MonsterStudio/releases/`，在单元格 1 填写 `RELEASE_ARCHIVE`（或打开同次生成的 `.ipynb`）。

代码解包或拉取至独立目录；不覆盖原有 `MonsterStudio/src`、模型、训练结果或项目数据。
升级时重新执行单元格 1 和 4 即可；有任务运行时禁止切换版本。

Worker 启动后会把地址、用户名和密码保存到 `MonsterStudio/runtime/endpoint_registry.json`。
本机客户端可用该私有文件连接；不要提交该文件到 Git。心跳会重置 600 秒空闲释放计时，不要持续轮询来保活。
批量任务原有的自动释放开关保留。这里的迁移不代表已完成 G4 实测或工作台制作面板接入。


In [ ]:
# @title 1. 安装并配置 Monster（默认从 GitHub 拉取，可选填 Drive 部署包）
import importlib.util, shutil, subprocess, sys, tarfile
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
GITHUB_REPO = "https://github.com/GrinZero/codex-game-studio.git" # @param {type:"string"}
GITHUB_REF = "main" # @param {type:"string"}
RELEASE_ARCHIVE = "" # @param {type:"string"}
ENABLE_DEPTH_VISIBILITY = True # @param {type:"boolean"}
ENABLE_2K_RESOLUTION = False # @param {type:"boolean"}
H3_PROFILE = "full" # @param ["full", "int8"]
CACHE_MODELS_ON_DRIVE = True # @param {type:"boolean"}

if RELEASE_ARCHIVE:
    archive = Path(RELEASE_ARCHIVE)
    if not archive.is_file():
        raise ValueError("请确认 RELEASE_ARCHIVE 指向已上传到 Drive 的 .tar.gz，或留空直接从 GitHub 拉取。")
    with tarfile.open(archive, "r:gz") as tar:
        installer = tar.extractfile("monster/scripts/install-release.py").read()
    installer_namespace = {"__name__": "monster_release_installer"}
    exec(compile(installer, "monster/scripts/install-release.py", "exec"), installer_namespace)
    MONSTER_PACKAGE = installer_namespace["install_release"](
        archive, "/content/monster-releases")
else:
    checkout = Path("/content/monster-releases/github-checkout")
    if checkout.exists():
        shutil.rmtree(checkout)
    checkout.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GITHUB_REF, GITHUB_REPO, str(checkout)], check=True)
    MONSTER_PACKAGE = checkout / "packages/monster"
spec = importlib.util.spec_from_file_location("monster_colab_runtime", MONSTER_PACKAGE / "scripts/colab_runtime.py")
monster_runtime = importlib.util.module_from_spec(spec)
spec.loader.exec_module(monster_runtime)
monster_runtime.configure(depth_visibility=ENABLE_DEPTH_VISIBILITY, enable_2k=ENABLE_2K_RESOLUTION,
                          profile=H3_PROFILE, cache_models=CACHE_MODELS_ON_DRIVE)


In [ ]:
# @title 2. 安装或复用 SkinTokens + Blender + FlashAttention
monster_runtime.setup_skintokens()


In [ ]:
# @title 3. 安装或复用 ComfyUI + MiniMax H3 及 Worker 依赖
monster_runtime.setup_comfy()


In [ ]:
# @title 4. 启动或热重载 Worker（输出地址，私有凭据保存到 Drive）
monster_runtime.launch_worker(port=7865)


In [ ]:
# @title 5. 任务进度（实时监控当前阶段、轮次、批次、loss、GPU 状态与预计剩余时间）
# 启动 Worker 后运行此单元格，每 5 秒自动刷新任务进度，单元格立即返回不阻塞运行。
monster_runtime.show_progress()


In [ ]:
# @title 6. 手动释放 GPU（点击后立即断开运行时）
import ipywidgets as widgets
from IPython.display import display
stop_btn = widgets.Button(description="立即释放 Colab G4 GPU", button_style="danger")
def on_stop(button):
    button.disabled = True
    monster_runtime.shutdown()
stop_btn.on_click(on_stop)
display(stop_btn)
